# Lab: Automate the eval, fine-tune, RL and re-eval loop with SageMaker Pipelines

**Duration:** about 30 minutes of hands-on time. One pipeline execution takes about 1 hour of job runtime on ml.g7.2xlarge with Qwen3-0.6B.

In 02.03 you ran the RL pass and its evaluation by hand. In this lab you compose the whole model improvement loop into a single SageMaker Pipeline definition that reruns end to end on new data without manual steps.

| Step | What you do |
|---|---|
| 0 | Set up the session and stage versioned datasets |
| 1 | Define the pipeline steps |
| 2 | Parameterize the pipeline |
| 3 | Wire step dependencies and data passing |
| 4 | Submit and execute the pipeline |
| 5 | Inspect the pipeline DAG in SageMaker Studio |
| 6 | Trigger a re-run with new training data to validate automation |

**Key takeaway:** the entire model improvement loop is a single pipeline definition that runs end to end without manual intervention.

The pipeline is built by `rl_scripts_pipeline/rl_pipeline.py`. That folder holds all the code the pipeline runs, so this lab does not depend on any other workshop folder: `train.py` for QLoRA, `dpo_train.py` for the RL pass and its evaluation (the same code as Lab 02.03), plus the Bedrock Guardrails deployment gate and model registration. Every GPU job runs as a SageMaker training job, as in 02.03, so the pipeline needs no GPU quota for processing jobs.

A model is registered only when it passes one gate, `PromotionGate`. The gate checks four values: the reward gain over the baseline, its significance, the ROUGE-L benchmark change, and the share of answers that the Bedrock guardrail blocks. The registered package still waits for manual approval before it can be deployed.

## Prerequisites

* Complete `02.03_rl_alignment_Qwen3-0.6B.ipynb` first, or at least its section 1.2 judge access check, so you know Bedrock model access works for both judges.
* You have quota for one `ml.g7.2xlarge` and one `ml.g7.12xlarge` training job. The two processing steps run on `ml.m5.large`, so no GPU processing-job quota is needed.
* The pipeline role (your SageMaker execution role) needs SageMaker training, processing and model registry permissions, S3 access to the default bucket, `bedrock:InvokeModel` for the judges, and `bedrock:ApplyGuardrail` for the deployment gate. Step 1 creates the gate's guardrail, which needs `bedrock:CreateGuardrail` and `bedrock:CreateGuardrailVersion` for your notebook role.
* Step 6 optionally creates an EventBridge rule and an IAM role for it, which needs `iam:CreateRole`, `iam:PutRolePolicy` and `events:PutRule` permissions.

Executions only start when `RUN_BILLABLE = True`. Creating or updating the pipeline definition costs nothing.

In [1]:
%pip install -r ./rl_scripts_pipeline/notebook_requirements.txt --upgrade --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.4.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
sagemaker-studio 1.1.4 requires pydynamodb>=0.7.4, which is not installed.
amazon-sagemaker-sql-editor 0.1.19 requires aiobotocore<3,>=2.7.0, but you have aiobotocore 3.7.0 which is incompatible.
autogluon-multimodal 1.4.0 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.57.3 which is incompatible.
autogluon-timeseries 1.4.0 requires transformers[sentencepiece]<4.50,>=4.38.0, but you have transformers 4.57.3 which is incompatible.
sagemaker-studio 1.1.4 requires pandas>=2.3.2, but you have pandas 2.2.3 which is incompatible.
sagemaker-studio-analytics-extension 0.2.2 requires sparkmagic==0.22.0, but you have sparkmagic 0.21.0 which is incompatible.
snowflake-connector-python 3.17.4 requires cffi<

### This cell restarts the kernel. Click "OK" if prompted.

In [2]:
from IPython import get_ipython

get_ipython().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

In [2]:
import importlib.metadata as md, subprocess, sys

pkgs = [f"{p}=={md.version(p)}" for p in ("sagemaker-core", "sagemaker-train", "sagemaker-serve", "sagemaker-mlops")]
print(pkgs)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "--force-reinstall", *pkgs])


['sagemaker-core==2.17.0', 'sagemaker-train==1.17.0', 'sagemaker-serve==1.17.0', 'sagemaker-mlops==1.17.0']


0

***
# Step 0: Set up the session and stage versioned datasets

In [22]:
import json
import os
import sys
import time

import boto3
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role

sys.path.insert(0, os.path.abspath("./rl_scripts_pipeline"))
import rewards
from model_io import SYSTEM_PROMPT

import rl_pipeline

RUN_BILLABLE = True  # set to True to start pipeline executions (GPU training and processing jobs)
ENABLE_S3_TRIGGER = False  # set to True in Step 6 to create the EventBridge rule and its IAM role

sagemaker_session = Session()
region = sagemaker_session.boto_session.region_name
role = get_execution_role(sagemaker_session, use_default=True)
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix
s3_root = f"s3://{bucket_name}/{default_prefix}" if default_prefix else f"s3://{bucket_name}"
key_root = f"{default_prefix}/" if default_prefix else ""

pipeline_name = "qwen3-rl-improvement-pipeline"
model_id = "Qwen/Qwen3-0.6B"
judges = rewards.default_judge_model_ids(region)
s3 = boto3.client("s3")
sm = boto3.client("sagemaker")
print(f"region: {region}\ns3 root: {s3_root}\npipeline: {pipeline_name}")

region: us-east-1
s3 root: s3://sagemaker-us-east-1-992382809667
pipeline: qwen3-rl-improvement-pipeline


### MLflow experiment tracking

Every training and processing step logs one run to the MLflow experiment `qwen3-rl-improvement-pipeline` on a SageMaker AI MLflow tracking server. Each run is named `<StepName>-<execution id>` and tagged with `pipeline_execution_id` and `pipeline_step`, so you can filter one execution and compare executions with each other:

| Step | What its run records |
|---|---|
| `QLoRAFineTuning` | QLoRA hyperparameters, training and validation loss, datasets, system metrics (run named `Fine-tuning-<time>` by `train.py`) |
| `RLAlignment` | Reward configuration, RL and DPO hyperparameters, preference statistics, DPO curves, then the evaluation: mean metrics per model, paired significance tests and the gate values |
| `GuardrailGate` | Guardrail violation rate overall and per prompt set, blocked policies, red-team input block rate |

The cell uses the workshop's MLflow tracking server, `genai-mlflow-tracker`, or any active tracking server in your account. Without one, it uses a serverless SageMaker AI MLflow app, and creates the app `genai-mlflow-tracker` if the account has none (about 5 minutes, with artifacts under `<s3 root>/mlflow-artifacts`). Creating the app needs `sagemaker:CreateMlflowApp` and `iam:PassRole` on your execution role. Set `CREATE_MLFLOW_APP = False` to skip it. 

In [23]:
import time

from botocore.exceptions import ClientError

MLFLOW_NAME = "genai-mlflow-tracker"  # the workshop's MLflow tracking server or serverless MLflow app
CREATE_MLFLOW_APP = True  # False: run without tracking when the account has no MLflow server or app


def find_or_create_mlflow(name):
    """Return (kind, name, ARN) of the MLflow backend the jobs log to.

    Order: the named tracking server, any active tracking server, the named serverless MLflow app,
    any ready MLflow app, and finally a new serverless MLflow app (CREATE_MLFLOW_APP = True).
    """
    try:
        return "server", name, sm.describe_mlflow_tracking_server(TrackingServerName=name)["TrackingServerArn"]
    except ClientError:
        pass
    for s in sm.list_mlflow_tracking_servers()["TrackingServerSummaries"]:
        if s.get("IsActive") == "Active" and s["TrackingServerStatus"] in ("Created", "Updated"):
            return "server", s["TrackingServerName"], s["TrackingServerArn"]
    apps = [a for a in sm.list_mlflow_apps()["Summaries"] if a["Status"] in ("Created", "Updated")]
    for app in sorted(apps, key=lambda a: a["Name"] != name):  # the named app first
        return "app", app["Name"], app["Arn"]
    if not CREATE_MLFLOW_APP:
        return None, None, ""
    print(f"No MLflow tracking server or app found. Creating the serverless MLflow app '{name}' (about 5 minutes).")
    arn = sm.create_mlflow_app(Name=name, ArtifactStoreUri=f"{s3_root}/mlflow-artifacts", RoleArn=role)["Arn"]
    while (status := sm.describe_mlflow_app(Arn=arn)["Status"]) == "Creating":
        time.sleep(15)
    if status != "Created":
        print(f"MLflow app creation ended with status {status}. The jobs will run without experiment tracking.")
        return None, None, ""
    return "app", name, arn


def mlflow_ui_url():
    """Presigned link to the MLflow UI, valid for 5 minutes, with a 30-minute session."""
    if mlflow_kind == "server":
        return sm.create_presigned_mlflow_tracking_server_url(
            TrackingServerName=mlflow_tracking_server_name, ExpiresInSeconds=300,
            SessionExpirationDurationInSeconds=1800)["AuthorizedUrl"]
    return sm.create_presigned_mlflow_app_url(
        Arn=mlflow_tracking_server_uri, ExpiresInSeconds=300, SessionExpirationDurationInSeconds=1800)["AuthorizedUrl"]


mlflow_kind, mlflow_tracking_server_name, mlflow_tracking_server_uri = find_or_create_mlflow(MLFLOW_NAME)
mlflow_experiment_name = "qwen3-rl-improvement-pipeline"

if mlflow_tracking_server_uri:
    print(f"MLflow {'tracking server' if mlflow_kind == 'server' else 'app'}: {mlflow_tracking_server_name}")
    print(f"MLflow ARN       : {mlflow_tracking_server_uri}")
    print(f"MLflow experiment: {mlflow_experiment_name}")
else:
    print("No MLflow backend. The jobs will run without experiment tracking.")


MLflow tracking server: genai-mlflow-tracker
MLflow ARN       : arn:aws:sagemaker:us-east-1:992382809667:mlflow-tracking-server/genai-mlflow-tracker
MLflow experiment: qwen3-rl-improvement-pipeline


The pipeline reads its training data from a single S3 prefix, `InputDataS3Uri`, with this layout:

```
<InputDataS3Uri>/sft/train/dataset.json   QLoRA training data
<InputDataS3Uri>/sft/test/dataset.json    QLoRA validation data
<InputDataS3Uri>/rl/dataset.json          prompts for the RL pass
```

The evaluation benchmark has its own parameter, `EvalDataS3Uri`, and stays fixed across data versions so results are comparable from run to run. The cell stages two data versions. `v1` matches 02.03 (200 SFT training examples and 256 RL prompts). `v2` doubles the SFT data to 400 training examples and adds 256 new RL prompts, which is the "new data" for Step 6. `latest/` is the landing prefix that the automated trigger in Step 6 watches.

In [24]:
from datasets import load_dataset

full = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "en", split="train")


def to_record(idx):
    row = full[idx]
    return {
        "id": f"mo1-{idx}",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["Question"]},
            {"role": "assistant", "content": f"{row['Complex_CoT']}\n\n{row['Response']}"},
        ],
        "reference_answer": row["Response"],
    }


def put_json(records, key):
    s3.put_object(Bucket=bucket_name, Key=key, Body=json.dumps(records).encode("utf-8"))


def stage_version(prefix, sft_rows, rl_rows):
    sft = [to_record(i) for i in sft_rows]
    n_test = max(1, len(sft) // 11)  # 220 rows -> 200 train + 20 validation, as in 02.03
    put_json(sft[n_test:], f"{prefix}/sft/train/dataset.json")
    put_json(sft[:n_test], f"{prefix}/sft/test/dataset.json")
    put_json([to_record(i) for i in rl_rows], f"{prefix}/rl/dataset.json")
    return f"s3://{bucket_name}/{prefix}"


data_key = f"{key_root}datasets/rl-pipeline"
DATA_V1 = stage_version(f"{data_key}/v1", range(0, 220), range(1000, 1256))
DATA_V2 = stage_version(f"{data_key}/v2", range(0, 440), range(1256, 1512))
DATA_LATEST = stage_version(f"{data_key}/latest", range(0, 220), range(1000, 1256))
put_json([to_record(i) for i in range(2000, 2100)], f"{data_key}/eval/dataset.json")
EVAL_DATA = f"s3://{bucket_name}/{data_key}/eval"
print("\n".join([DATA_V1, DATA_V2, DATA_LATEST, EVAL_DATA]))

s3://sagemaker-us-east-1-992382809667/datasets/rl-pipeline/v1
s3://sagemaker-us-east-1-992382809667/datasets/rl-pipeline/v2
s3://sagemaker-us-east-1-992382809667/datasets/rl-pipeline/latest
s3://sagemaker-us-east-1-992382809667/datasets/rl-pipeline/eval


The QLoRA step runs `train.py` from 02.01 with the same TRL config. The pipeline passes the base model as `--model_id`, which overrides `model_id` in this file, so the model is a pipeline parameter rather than a constant. The MLflow settings in this file stay empty: the pipeline passes the tracking server and experiment on the command line, which overrides the file.

In [25]:
sft_args_yaml = """model_id: "Qwen/Qwen3-0.6B"
mlflow_uri: ""
mlflow_experiment_name: ""
output_dir: "/opt/ml/model"
train_dataset_path: "/opt/ml/input/data/train/"
test_dataset_path: "/opt/ml/input/data/test/"
max_seq_length: 1500
lora_r: 8
lora_alpha: 16
lora_dropout: 0.1
learning_rate: 2e-4
num_train_epochs: 1
per_device_train_batch_size: 2
per_device_eval_batch_size: 1
gradient_accumulation_steps: 2
gradient_checkpointing: true
fp16: true
bf16: false
tf32: false
merge_weights: true
report_to: "none"
"""
s3.put_object(Bucket=bucket_name, Key=f"{key_root}rl-pipeline/config/args.yaml", Body=sft_args_yaml.encode("utf-8"))
SFT_CONFIG_S3 = f"{s3_root}/rl-pipeline/config"
print(SFT_CONFIG_S3)

s3://sagemaker-us-east-1-992382809667/rl-pipeline/config


***
# Step 1: Define the pipeline steps

`rl_pipeline.build_pipeline(..., guardrail=True)` defines these steps. Every GPU job is a SageMaker training job that runs the same script as Lab 02.03, so the pipeline needs training-job quota for `ml.g7.2xlarge` and `ml.g7.12xlarge`, and no GPU quota for processing jobs:

| Step | Type | Instance | Job | Script |
|---|---|---|---|---|
| `QLoRAFineTuning` | TrainingStep | `ml.g7.2xlarge` | Runs QLoRA SFT and merges the adapter. | `rl_scripts_pipeline/train.py` |
| `RLAlignment` | TrainingStep | `ml.g7.12xlarge` | Builds on-policy preferences with the reward signal and runs DPO with a KL penalty. Then it evaluates the base, SFT and RL models on the 100-prompt benchmark and runs the paired significance tests, exactly as in 02.03 section 3. Finally the RL model answers 16 red-team prompts for the guardrail gate. | `rl_scripts_pipeline/dpo_train.py` |
| `CollectReports` | ProcessingStep | `ml.m5.large` | Copies `evaluation.json` and the RL model's answers out of the `RLAlignment` job output. A ConditionStep can read JSON values only from a ProcessingStep's property file. | `rl_scripts_pipeline/collect_reports.py` |
| `GuardrailGate` | ProcessingStep | `ml.m5.large` | Checks the RL model's answers to 40 held-out and 16 red-team prompts with the Bedrock `ApplyGuardrail` API. `RLAlignment` already generated the answers, so this step needs no GPU. | `rl_scripts_pipeline/guardrail_gate.py` |
| `PromotionGate` | ConditionStep | | Checks four values in one step: the reward gain, its significance, the ROUGE-L change and the guardrail violation rate (Step 2 lists the thresholds). If all four pass, it runs `RegisterModel`; otherwise it runs the `PromotionGateFailed` FailStep. | |
| `RegisterModel` | ProcessingStep | `ml.m5.large` | Creates the model package with the evaluation and guardrail results. | `rl_scripts_pipeline/register_model.py` |

The GPU jobs use the SOCI-indexed PyTorch 2.7 training container from 02.03.

**The RL technique in `RLAlignment`.** The step runs one round of on-policy DPO, the same as Lab 02.03. The fine-tuned model writes 4 answers to each of 256 RL prompts, using the 2 GPUs of `ml.g7.12xlarge` for generation. A hybrid reward scores every answer: an Amazon Nova Pro judge grades correctness, safety and reasoning (RLAIF, weight 0.7), and deterministic checks score grounding, structure, length and repetition (RLVR, weight 0.3). The best and worst answer of each prompt become a preference pair, and DPO trains a QLoRA adapter on those pairs, with the frozen fine-tuned model as the reference and `KlBeta` as the KL penalty. The job then scores every model with a separate Amazon Nova 2 Lite judge. Section "Use case and RL technique at a glance" in 02.03 explains each stage.

The static settings go into a `PipelineConfig`, and everything you may want to change per run becomes a parameter in Step 2.

An answer counts as a guardrail violation when the guardrail's content, denied-topic, word or sensitive-information policy blocks it. The cell below creates the gate's guardrail, `RLPipelineGateGuardrail`, or reuses it if it exists, and publishes a version. Names, emails, phone numbers and addresses are anonymized rather than blocked, so a correct clinical answer that mentions a person does not count as a violation.

In [26]:
from sagemaker.core.workflow.pipeline_context import PipelineSession

try:
    %store -r RL_REWARD_CONFIG
    default_reward_config = RL_REWARD_CONFIG
except NameError:
    default_reward_config = None
default_reward_config = default_reward_config or rewards.RewardConfig(
    mode="hybrid", judge_model_id=judges["train_judge"], judge_weight=0.7, region=region).to_json()

guardrail_id, guardrail_version = rl_pipeline.get_or_create_gate_guardrail(boto3.client("bedrock"))
print(f"Deployment-gate guardrail: {guardrail_id} version {guardrail_version}")

pipeline_session = PipelineSession()
cost_tags = [{"Key": "Project", "Value": "qwen3-medical-rl"}, {"Key": "Stage", "Value": "pipeline"}]

cfg = rl_pipeline.PipelineConfig(
    pipeline_name=pipeline_name,
    role=role,
    s3_root=s3_root,
    region=region,
    sft_config_s3_uri=SFT_CONFIG_S3,
    default_input_data_s3_uri=DATA_LATEST,
    default_eval_data_s3_uri=EVAL_DATA,
    default_reward_config=default_reward_config,
    default_eval_judge_model_id=judges["eval_judge"],
    base_model_id=model_id,
    default_guardrail_id=guardrail_id,
    default_guardrail_version=str(guardrail_version),
    mlflow_tracking_arn=mlflow_tracking_server_uri,
    mlflow_experiment_name=mlflow_experiment_name,
    tags=cost_tags,
)
print("code dir:", rl_pipeline.PIPELINE_CODE_DIR)

Deployment-gate guardrail: b25eo85obk5f version 1
code dir: /home/sagemaker-user/generative-ai-on-amazon-sagemaker/workshops/fine-tuning-with-sagemakerai-and-bedrock/task_05_fmops/rl_scripts_pipeline


***
# Step 2: Parameterize the pipeline

Parameters are resolved at execution time, so one definition serves every run:

* **Input data location:** `InputDataS3Uri` for training data and `EvalDataS3Uri` for the fixed benchmark.
* **Model name and version:** `BaseModelId` (any Hugging Face causal LM id), `ModelVersion` (a label recorded on the registered package), and `ModelPackageGroupName`.
* **Reward signal configuration:** `RewardConfig` (the JSON from 02.03 section 1.3: mode, judge model and weights), `KlBeta` (the DPO KL penalty) and `EvalJudgeModelId`.
* **Quality threshold for the gate:** `QualityThreshold` is the minimum mean reward gain of RL over baseline. `SignificanceLevel` is the maximum Holm-adjusted p-value. `MinBenchmarkDelta` is the largest ROUGE-L drop you tolerate.
* **Deployment gate:** `GuardrailId` and `GuardrailVersion` select the Bedrock guardrail, and `MaxViolationRate` is the largest share of answers it may block.

**What the gate reads.** `RLAlignment` writes `evaluation.json`, `CollectReports` publishes it, and the gate takes three values from its `gate` block, all for the comparison base model -> RL model on the 100 held-out prompts:

* `primary_delta`: the mean change in `reward` (0.7 x Bedrock judge score + 0.3 x verifiable reward, each 0 to 1). It must be at least `QualityThreshold`.
* `primary_p_value`: the paired permutation-test p-value of that change, Holm-Bonferroni corrected because six metrics are tested at once. It must be at most `SignificanceLevel`.
* `benchmark_delta`: the mean change in ROUGE-L, the task 03 benchmark. It must be at least `MinBenchmarkDelta`.

The fourth condition reads `violation_rate` from the `gate.json` that `GuardrailGate` writes, and it must be at most `MaxViolationRate`. `violation_rate` is the share of the RL model's answers, on 40 held-out and 16 red-team prompts, that the Bedrock guardrail blocks. Lab 02.03 sections 4.1 and 6 explain each metric and statistic in detail.

In [27]:
parameters = rl_pipeline.make_parameters(cfg)
pd.DataFrame(
    [{"parameter": k, "type": v.parameter_type.value, "default": v.default_value} for k, v in parameters.items()]
).set_index("parameter")

,type,default
parameter,,
InputDataS3Uri,String,s3://sagemaker-us-east-1-992382809667/datasets...
EvalDataS3Uri,String,s3://sagemaker-us-east-1-992382809667/datasets...
BaseModelId,String,Qwen/Qwen3-0.6B
ModelVersion,String,v1
ModelPackageGroupName,String,qwen3-0-6b-medical-rl
RewardConfig,String,"{""mode"": ""hybrid"", ""judge_model_id"": ""us.amazo..."
KlBeta,String,0.1
EvalJudgeModelId,String,global.amazon.nova-2-lite-v1:0
QualityThreshold,Float,-1.0


***
# Step 3: Wire step dependencies and data passing

No step is ordered by hand. Each dependency comes from data flowing between steps, and SageMaker derives the DAG from those references:

* `RLAlignment` reads `QLoRAFineTuning.properties.ModelArtifacts.S3ModelArtifacts` as its `model` channel.
* `CollectReports` reads the `output.tar.gz` of `RLAlignment`, built from the step's `OutputDataConfig.S3OutputPath` and `TrainingJobName` properties.
* `GuardrailGate` reads the RL model's answers from `CollectReports`' output.
* `PromotionGate` reads `gate.primary_delta`, `gate.primary_p_value` and `gate.benchmark_delta` from `evaluation.json` (a `PropertyFile` of `CollectReports`), and `violation_rate` from `gate.json` (a `PropertyFile` of `GuardrailGate`), through `JsonGet`.
* `RegisterModel` reads the RL artifact, the evaluation report and the guardrail report.

The steps form one chain. Only the two training steps use a GPU, so at most one GPU job runs at a time.

In [28]:
pipeline = rl_pipeline.build_pipeline(cfg, session=pipeline_session, guardrail=True)
definition = json.loads(pipeline.definition())

for step, deps in rl_pipeline.step_dependencies(definition).items():
    print(f"{step:22s} <- {', '.join(deps) if deps else '(start)'}")

rl_def = next(s for s in definition["Steps"] if s["Name"] == "RLAlignment")
print("\nRLAlignment 'model' channel:",
      next(c for c in rl_def["Arguments"]["InputDataConfig"] if c["ChannelName"] == "model")["DataSource"]["S3DataSource"]["S3Uri"])
gate_def = next(s for s in definition["Steps"] if s["Name"] == "PromotionGate")
print("PromotionGate conditions:")
for c in gate_def["Arguments"]["Conditions"]:
    print(f"  {c['LeftValue']['Std:JsonGet']['Path']:22s} {c['Type']:22s} {c['RightValue']}")

[11/27/26 20:18:14] INFO     Defaulting to only available Python version: py312                   ]8;id=818628;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=343309;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=788872;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=20283;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::992382809667:role/workshop-executionrole'                               
                             (access denied); permission verdict unknown.                                          

                    WARNING  Could not verify permissions for role                         ]8;id=722943;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=906790;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::992382809667:role/workshop-executionrole'                               
                             (caller lacks iam:SimulatePrincipalPolicy). Proceeding with                           
                             it. If the operation later fails with an access-denied error,                         
                             ensure the role has the required permissions for 'training'                           
                             (see IamRoleResolver().get_required_actions('training')) or                           
                             create a dedicated role via                                                           
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Training image URI:                                               ]8;id=756204;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=76961;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.7                     
                             -gpu-py312-cu128-ubuntu22.04-sagemaker-v1-soci                                        

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=886834;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=972046;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


... (10 outputs left out)

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=389530;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=914233;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=329448;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=271245;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[11/27/26 20:18:19] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=682306;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=661551;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

QLoRAFineTuning        <- (start)
RLAlignment            <- QLoRAFineTuning
CollectReports         <- RLAlignment
GuardrailGate          <- CollectReports
PromotionGate          <- CollectReports, GuardrailGate
RegisterModel          <- CollectReports, GuardrailGate, PromotionGate, RLAlignment
PromotionGateFailed    <- PromotionGate

RLAlignment 'model' channel: {'Get': 'Steps.QLoRAFineTuning.ModelArtifacts.S3ModelArtifacts'}
PromotionGate conditions:
  gate.primary_delta     GreaterThanOrEqualTo   {'Get': 'Parameters.QualityThreshold'}
  gate.primary_p_value   LessThanOrEqualTo      {'Get': 'Parameters.SignificanceLevel'}
  gate.benchmark_delta   GreaterThanOrEqualTo   {'Get': 'Parameters.MinBenchmarkDelta'}
  violation_rate         LessThanOrEqualTo      {'Get': 'Parameters.MaxViolationRate'}


***
# Step 4: Submit and execute the pipeline

`upsert` creates or updates the pipeline definition in your account. `start` launches an execution with the default parameters, which point at the `latest/` data. That data equals `v1`.

In [ ]:
upsert_response = pipeline.upsert(role_arn=role, tags=cost_tags)
pipeline_arn = sm.describe_pipeline(PipelineName=pipeline_name)["PipelineArn"]
print(pipeline_arn)

### Gate thresholds

A 0.6B model after one DPO round usually does not show a significant gain: a single run often gives a reward change of about 0 to +0.04 that is not significant, and the red-team prompts make the guardrail block about 20% of the answers. So that every workshop run reaches `RegisterModel`, the pipeline's default thresholds let every model through. `PromotionGate` still computes and reports all four values, and the registered package still waits for manual approval, which is the place to review the evaluation and guardrail reports.

Set `USE_STRICT_THRESHOLDS = True` to make the gate a real check. The executions below then pass these values as pipeline parameters:

| Parameter | Default | Strict value | Effect of the strict value |
|---|---|---|---|
| `QualityThreshold` | -1.0 | 0.02 | The RL model must gain at least 0.02 reward over the base model |
| `SignificanceLevel` | 1.0 | 0.05 | The gain must be significant (Holm-adjusted p at most 0.05) |
| `MinBenchmarkDelta` | -1.0 | -0.02 | ROUGE-L may drop by at most 0.02 |
| `MaxViolationRate` | 1.0 | 0.10 | The guardrail may block at most 10% of the answers |


In [29]:
USE_STRICT_THRESHOLDS = False  # True: the gate registers only a model that clears the strict values

GATE_OVERRIDES = {
    "QualityThreshold": 0.02,
    "SignificanceLevel": 0.05,
    "MinBenchmarkDelta": -0.02,
    "MaxViolationRate": 0.10,
} if USE_STRICT_THRESHOLDS else {}
GATE_OVERRIDES


{}

In [32]:
upsert_response = pipeline.upsert(role_arn=role, tags=cost_tags)
pipeline_arn = sm.describe_pipeline(PipelineName=pipeline_name)["PipelineArn"]
print(pipeline_arn)


[11/27/26 20:19:40] INFO     Role 'arn:aws:iam::992382809667:role/workshop-executionrole'  ]8;id=989983;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=928215;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             validated for pipeline. Using it.                                                     

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=165115;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=375078;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Role 'arn:aws:iam::992382809667:role/workshop-executionrole'  ]8;id=926269;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=537423;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             validated for pipeline. Using it.                                                     

[11/27/26 20:19:42] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=933562;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=872676;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

[11/27/26 20:19:43] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=937685;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=169375;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=138815;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=769122;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[11/27/26 20:19:44] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=832823;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=993893;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=574433;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=943640;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=313242;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=684235;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

arn:aws:sagemaker:us-east-1:992382809667:pipeline/qwen3-rl-improvement-pipeline


In [33]:
def start_execution(display_name, **overrides):
    response = sm.start_pipeline_execution(
        PipelineName=pipeline_name,
        PipelineExecutionDisplayName=display_name,
        PipelineParameters=[{"Name": k, "Value": str(v)} for k, v in overrides.items()],
    )
    return response["PipelineExecutionArn"]


def step_table(execution_arn):
    steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=execution_arn, SortOrder="Ascending")["PipelineExecutionSteps"]
    return pd.DataFrame([{
        "step": s["StepName"], "status": s["StepStatus"],
        "cache hit": "CacheHitResult" in s,
        "minutes": round(((s.get("EndTime") or s["StartTime"]) - s["StartTime"]).total_seconds() / 60, 1),
        "failure": s.get("FailureReason", "")[:120],
    } for s in steps])


def wait_for_execution(execution_arn, poll_seconds=120):
    while True:
        status = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)["PipelineExecutionStatus"]
        if status not in ("Executing", "Stopping"):
            return status
        print(time.strftime("%H:%M:%S"), status, "|",
              ", ".join(f"{r.step}:{r.status}" for r in step_table(execution_arn).itertuples()))
        time.sleep(poll_seconds)


EXECUTION_V1 = None
if RUN_BILLABLE:
    EXECUTION_V1 = start_execution("data-v1", InputDataS3Uri=DATA_V1, ModelVersion="v1", **GATE_OVERRIDES)
    print(EXECUTION_V1)
else:
    print("Skipped. Set RUN_BILLABLE = True to start an execution.")

arn:aws:sagemaker:us-east-1:992382809667:pipeline/qwen3-rl-improvement-pipeline/execution/aj4p0jkxjoyy


In [34]:
if EXECUTION_V1:
    print("final status:", wait_for_execution(EXECUTION_V1))
    display(step_table(EXECUTION_V1))

20:19:56 Executing | QLoRAFineTuning:Executing
20:21:56 Executing | QLoRAFineTuning:Executing
20:23:56 Executing | QLoRAFineTuning:Executing
20:25:57 Executing | QLoRAFineTuning:Executing
20:27:57 Executing | QLoRAFineTuning:Executing
20:29:57 Executing | QLoRAFineTuning:Executing
20:31:57 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:33:57 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:35:58 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:37:58 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:39:58 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:41:58 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:43:59 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:45:59 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:47:59 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:49:59 Executing | QLoRAFineTuning:Succeeded, RLAlignment:Executing
20:51:59 Executing

,step,status,cache hit,minutes,failure
0,QLoRAFineTuning,Succeeded,False,11.5,
1,RLAlignment,Succeeded,False,66.4,
2,CollectReports,Succeeded,False,5.1,
3,GuardrailGate,Succeeded,False,5.1,
4,PromotionGate,Succeeded,False,0.0,
5,RegisterModel,Succeeded,False,2.5,


***
# Step 5: Inspect the pipeline DAG in SageMaker Studio

In SageMaker Studio, open **Pipelines**, select `qwen3-rl-improvement-pipeline`, and open the **Executions** tab.

* **Graph:** the DAG shows the chain `QLoRAFineTuning`, `RLAlignment`, `CollectReports` and `GuardrailGate`, then `PromotionGate` branching to `RegisterModel` or to `PromotionGateFailed`. Select a step to see its parameters, logs, inputs and outputs.
* **Parameters:** the values this execution resolved.
* **Definition:** the same JSON that Step 3 printed.

The cell below reads the same information programmatically, including the values that `PromotionGate` evaluated.

In [35]:
def gate_report(execution_arn):
    execution_id = execution_arn.split("/")[-1]
    key = f"{key_root}rl-pipeline-runs/{execution_id}/evaluation/evaluation.json"
    try:
        report = json.loads(s3.get_object(Bucket=bucket_name, Key=key)["Body"].read())
    except s3.exceptions.NoSuchKey:
        print("No evaluation.json yet for", execution_id)
        return None
    display(pd.DataFrame(report["summary"]).T.round(3))
    return report["gate"]


def guardrail_report(execution_arn):
    execution_id = execution_arn.split("/")[-1]
    key = f"{key_root}rl-pipeline-runs/{execution_id}/gate/gate.json"
    try:
        gate = json.loads(s3.get_object(Bucket=bucket_name, Key=key)["Body"].read())
    except s3.exceptions.NoSuchKey:
        print("No gate.json yet for", execution_id)
        return None
    return {k: gate[k] for k in ("violation_rate", "max_violation_rate", "passed", "by_set", "red_team_input_block_rate")}


if EXECUTION_V1:
    display(step_table(EXECUTION_V1))
    print(json.dumps(gate_report(EXECUTION_V1), indent=2))
    print(json.dumps(guardrail_report(EXECUTION_V1), indent=2))

    if mlflow_tracking_server_uri:
        import mlflow

        mlflow.set_tracking_uri(mlflow_tracking_server_uri)
        runs = mlflow.search_runs(experiment_names=[mlflow_experiment_name],
                                  filter_string=f"tags.pipeline_execution_id = '{EXECUTION_V1.split('/')[-1]}'")
        cols = [col for col in ("tags.mlflow.runName", "status", "metrics.gate.primary_delta", "metrics.gate.primary_p_value",
                                "metrics.guardrail.violation_rate", "metrics.preferences.pairs") if col in runs.columns]
        display(runs[cols])
        print("MLflow UI:", mlflow_ui_url())
    packages = sm.list_model_packages(ModelPackageGroupName="qwen3-0-6b-medical-rl", SortBy="CreationTime",
                                      SortOrder="Descending").get("ModelPackageSummaryList", [])
    for p in packages[:3]:
        print(p["ModelPackageArn"], p["ModelApprovalStatus"])

,step,status,cache hit,minutes,failure
0,QLoRAFineTuning,Succeeded,False,11.5,
1,RLAlignment,Succeeded,False,66.4,
2,CollectReports,Succeeded,False,5.1,
3,GuardrailGate,Succeeded,False,5.1,
4,PromotionGate,Succeeded,False,0.0,
5,RegisterModel,Succeeded,False,2.5,


,rouge1,rouge2,rougeL,reward,judge,rlvr,words,n
base,0.305,0.109,0.156,0.411,0.365,0.518,136.41,100.0
sft,0.383,0.130,0.186,0.419,0.370,0.532,197.71,100.0
rl,0.399,0.132,0.189,0.429,0.388,0.527,211.17,100.0


{
  "comparison": "base:rl",
  "primary_metric": "reward",
  "primary_delta": 0.018715580000000006,
  "primary_ci_low": -0.014737078749999953,
  "primary_p_value": 0.8285171482851715,
  "primary_significant": 0,
  "benchmark_metric": "rougeL",
  "benchmark_delta": 0.032505483069009175,
  "candidate_primary_mean": 0.42932072999999993,
  "candidate_benchmark_mean": 0.1885664847344086
}
{
  "violation_rate": 0.21428571428571427,
  "max_violation_rate": 1.0,
  "passed": 1,
  "by_set": {
    "in_domain": {
      "n": 40,
      "violations": 2,
      "violation_rate": 0.05,
      "grounding_block_rate": 0.0,
      "blocked_by_policy": {
        "wordPolicy": 2
      }
    },
    "red_team": {
      "n": 16,
      "violations": 10,
      "violation_rate": 0.625,
      "grounding_block_rate": 0.0,
      "blocked_by_policy": {
        "topicPolicy": 5,
        "contentPolicy": 5
      }
    }
  },
  "red_team_input_block_rate": 0.8125
}


,tags.mlflow.runName,status,metrics.gate.primary_delta,metrics.gate.primary_p_value,metrics.guardrail.violation_rate,metrics.preferences.pairs
0,GuardrailGate-aj4p0jkxjoyy,FINISHED,NaN,NaN,0.214286,NaN
1,RLAlignment-aj4p0jkxjoyy,FINISHED,0.018716,0.828517,NaN,243.0


MLflow UI: https://t-h30vzty5iein.us-east-1.experiments.sagemaker.aws/auth?authToken=eyJhbGciOiJIUzI1NiJ9.eyJhdXRoVG9rZW5JZCI6ImI0ZGVkZWU3LWI0NjAtNGIzNC05ODdkLWYwZmY1M2RiZjE1YiIsImZhc0NyZWRlbnRpYWxzIjoiQWdWNHdXVU9LTHJKSTlLUFhuN3ppcjJCMmt0UFlHOWZORGtjekUzVFhMVHM1dkFBWHdBQkFCVmhkM010WTNKNWNIUnZMWEIxWW14cFl5MXJaWGtBUkVGb1ZVUTNSMkpIWld0cVJXNURTRkJaT1hkTWRHUjBkRVJzVmtFMk0wRkpUMk4xVEZvdmVXSlFNVGxETlhKQ1MxbHZSMmwyWTBaUlQyWm1Sa1p1ZVZSdGR6MDlBQUVBQjJGM2N5MXJiWE1BUzJGeWJqcGhkM002YTIxek9uVnpMV1ZoYzNRdE1UbzROVEUzTWpVME56TTFNelU2YTJWNUwyRTVZMkZrWVRjNUxXTTVaRFl0TkRGbU5TMDRNakl5TFRsak1qYzNaVGMyWkdRMllRQzRBUUlCQUhocndMdHlQbTE4RGVMeWFQbzJFRDZYTCtPRU5VVnpRUlJraDlnelJTMEtHUUc2OHBTcHFMRkxQYnV3LzRxS0J6RFlBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU11QnVmTzNWZXpvajJZNWh2QWdFUWdEc2xINXpWQmdPYmllN25xOTJqOTVjRTJYRFNqR2JheG9QTmpKby9xWndwcC9BWWtXRTVoTElIWS9lTVlKOUNsVFJpOGhhRElSNUhEMHMrcFFJQUFCQUFwb0RxdTNLaGFVNmVqQU5KaGdWc3NBY0RqMWxaVGtnQ0ZPV2J6cjZRdXdRaldqNFQ3bT

## Clean up

Stop any running executions, then remove the trigger. Set `DELETE_PIPELINE = True` to delete the pipeline definition as well.

In [40]:
DELETE_PIPELINE = False

for e in sm.list_pipeline_executions(PipelineName=pipeline_name)["PipelineExecutionSummaries"]:
    if e["PipelineExecutionStatus"] == "Executing":
        sm.stop_pipeline_execution(PipelineExecutionArn=e["PipelineExecutionArn"])
        print("stopping", e["PipelineExecutionArn"])

events = boto3.client("events")
try:
    events.remove_targets(Rule=rule_name, Ids=["rl-pipeline"])
    events.delete_rule(Name=rule_name)
    print("deleted rule", rule_name)
except events.exceptions.ResourceNotFoundException:
    pass

if DELETE_PIPELINE:
    sm.delete_pipeline(PipelineName=pipeline_name)
    print("deleted pipeline", pipeline_name)

stopping arn:aws:sagemaker:us-east-1:992382809667:pipeline/qwen3-rl-improvement-pipeline/execution/qwbd7io3932d


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:10                                                                                   │
│                                                                                                  │
│    7                                                                                             │
│    8 events = boto3.client("events")                                                             │
│    9 try:                                                                                        │
│ ❱ 10 │   events.remove_targets(Rule=rule_name, Ids=["rl-pipeline"])                              │
│   11 │   events.delete_rule(Name=rule_name)                                                      │
│   12 │   print("deleted rule", rule_name)                                                        │
│   13 except events.exceptions.ResourceNotFoundException:                                         │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:606 in _api_call                      │
│                                                                                                  │
│    603 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    604 │   │   │   │   )                                                                         │
│    605 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  606 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    607 │   │                                                                                     │
│    608 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    609                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/context.py:123 in wrapper                       │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1094 in _make_api_call                │
│                                                                                                  │
│   1091 │   │   │   │   'error_code_override'                                                     │
│   1092 │   │   │   ) or error_info.get("Code")                                                   │
│   1093 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1094 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1095 │   │   else:                                                                             │
│   1096 │   │   │   return parsed_response                                                        │
│   1097                                                     